# Cross-kill — does T-cell + NK + bystander wave close the MHC blind spot? (RUNG-21)

**CPU only — no GPU, ~1 GB RAM, ~3–7 min.** The operational close of RUNG-18/18b/19.

Real tumours carry MHC-dark 'escapee' cells (~4–13%, measured). T-cells can't see them. But **NK cells kill cells that LOST MHC** ('missing-self') — the opposite trigger. So the tumour is trapped: keep MHC → T kills; drop MHC → NK kills. This tests whether layered **T + NK + bystander wave** clears what T-cells alone can't.

**Set Runtime → CPU.**

In [ ]:
#@title Cell 1 — clone / pull
import os
from pathlib import Path
REPO=Path('/content/cancer-recog-apoptosis')
if REPO.exists():
    !cd {REPO} && rm -f runs/logs/*.log && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/AnshumanAtrey/cancer-recog-apoptosis.git {REPO}
os.chdir(REPO)
!git log --oneline -1
print('[CELL 1] ✓')

In [ ]:
#@title Cell 2 — run log + VALIDATE (selftest)
import sys
from scripts.runlog import new_log, run_logged
RUNLOG=new_log('rung21_crosskill', repo_dir='.')
rc=run_logged([sys.executable,'-u','scripts/47_crosskill.py','selftest'], RUNLOG)
print('[CELL 2]', '✓ validated' if rc==0 else f'✗ FAILED ({rc}) — stop')

In [ ]:
#@title Cell 3 — RUN cross-kill  [CPU, ~3-7 min]
mode='run' #@param ['run','quick']
import sys, os, json
from scripts.runlog import run_logged
run_logged([sys.executable,'-u','scripts/47_crosskill.py', mode], RUNLOG)
from IPython.display import Image, display
p='runs/rung21_crosskill/rung21_crosskill'
if os.path.exists(p+'.png'): display(Image(p+'.png'))
if os.path.exists(p+'.json'):
    d=json.load(open(p+'.json'))
    print('HEADLINE:', d['HEADLINE']['plain'])
    print('\nP(cure) vs escapee fraction:', json.dumps(d['p_cure_vs_escapee_fraction'], indent=2))

In [ ]:
#@title Cell 4 — bundle zip + download
import os, glob, zipfile
from scripts.runlog import finalize
finalize(RUNLOG, download=False)
stem=os.path.basename(str(RUNLOG)).replace('rung21_crosskill_','').replace('.log','')
b=f'/content/rung21_crosskill_{stem}.zip'
ps=sorted(glob.glob('runs/rung21_crosskill/*.png')+glob.glob('runs/rung21_crosskill/*.json')+[str(RUNLOG)])
with zipfile.ZipFile(b,'w',zipfile.ZIP_DEFLATED) as z:
    for x in ps:
        if os.path.exists(x): z.write(x, arcname=os.path.basename(x)); print(' bundled', x)
print('[CELL 4] ->', b)
try:
    from google.colab import files; files.download(b)
except Exception as e: print('(skipped:', e, ')')